In [1]:
# Import libraries
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import pchip_interpolate
from scipy.optimize import minimize 

# Define default plotting parameters
plt.rcParams['font.size'] = '10'  
plt.rcParams['savefig.dpi'] = 300  

# Import local module for genetic algorithm
from genetic_algorithm_pfm import GeneticAlgorithm 

In [ ]:
# Define the names of variables for later use in plotting and analysis
design_variables = (
    ('x1', 'Bridge width',        'motorized vehicle lanes'),
    ('x2', 'Vertical clearance',  'm'),
)

# set bounds for all variables
b1 = [2, 8]      # x1
b2 = [9.25, 14.5]      # x2
bounds = [b1, b2]

X_irl = [4, 11.4]
plot_irl = True  # Can be true or False, depending on whether you want to plot the IRL point or not

In [12]:
# Define objective functions

def objective_function_1(x1, x2):
    """
    Cost function.
    
    :return: float of construction cost in euros.
    """
    cons1 = 72500000           # costs per lane
    cons2 = 34100000/1.35      # cost per metre vertical clearance

    if x1 <= 4 and x2 <= 11.4:
        return 4*cons1 - cons1 * x1 + 11.4 * cons2 - cons2 * x2
    elif x1 <= 4 and x2 > 11.4:
        return 4*cons1 - cons1 * x1 - 11.4 * cons2 + cons2 * x2
    elif x1 > 4 and x2 <= 11.4:
        return -4*cons1 + cons1 * x1 + 11.4 * cons2 - cons2 * x2
    elif x1 > 4 and x2 > 11.4:
        return -4*cons1 + cons1 * x1 - 11.4 * cons2 + cons2 * x2


def objective_function_2(x1, x2):
    """
    Water Pollution function.
    
    :return: float of water pollution in ships/hour.
    """
    x = np.array([9.25, 10.5, 11.5, 12.5, 13.5, 14.5])
    y = np.array([0.42, 0.5785, 0.62575, 0.649375, 0.661, 0.679])
    coefficienten = np.polyfit(x, y, 2)
    f = np.poly1d(coefficienten)

    return f(x2) * 15


def objective_function_3(x1, x2):
    """
    Shipping flow function.
    
    :return: float of shipping flow in ships/hour.
    """
    x = np.array([9.25, 10.5, 11.5, 12.5, 13.5, 14.5])
    y = np.array([0.42, 0.5785, 0.62575, 0.649375, 0.661, 0.679])
    coefficienten = np.polyfit(x, y, 2)
    f = np.poly1d(coefficienten)
    
    return f(x2) * 15


def objective_function_4(x1, x2):
    """
    Disturbance function.
    
    :return: float of disturbance in terms of expected number of complaints per year.
    """
    cons13 = 1/(24*800)     # complaints per vehicle
    cons14 = 4.8/78 * 1/(24*15)     # complaints per ship
    cons15 = 800     # vehicles/hour/4 lanes
    x = np.array([9.25, 10.5, 11.5, 12.5, 13.5, 14.5])
    y = np.array([0.42, 0.5785, 0.62575, 0.649375, 0.661, 0.679])
    coefficienten = np.polyfit(x, y, 2)
    f = np.poly1d(coefficienten)

    return (cons13 * x1/4 * cons15 + cons14 * f(x2) * 15)*24*365


def objective_function_5(x1, x2):
    """
    Traffic flow function.
    
    :return: float of predicted traffic flow in vehicles/hour.
    """
    cons16 = 800     # vehicles/hour/4 lanes
    return x1/4 * cons16


# Define the list of objectives with their corresponding names and units and stakeholders for later use in plotting and analysis
objectives = [
    (objective_function_1, "Cost",                   "€",                                "Municipality of Rotterdam"),
    (objective_function_2, "Water Pollution",        "ships/hour",                       "Rijkswaterstaat"),
    (objective_function_3, "Shipping flow",          "ships/hour",                       "Port of Rotterdam"),
    (objective_function_4, "Disturbance",            "complaints/year",                  "Inhabitants"),
    (objective_function_5, "Traffic flow",           "vehicles/hour",                    "Road users")
]


In [13]:
# Finding min and max for each objective using scipy's minimize function, starting from the midpoint of the bounds
objective_minmax = {} # Dictionary to store the min and max values for each objective       
midpoints = [np.mean(b) for b in bounds]

for idx, (obj_func, name, unit, stakeholder) in enumerate(objectives):
    wrapped = lambda x, sign=1: sign * obj_func(*x)  # obj_func accepts a single array-like X
    
    min_val =  minimize(wrapped, x0=midpoints, bounds=bounds, method='L-BFGS-B').fun
    max_val = -minimize(lambda x: wrapped(x, sign=-1), x0=midpoints, bounds=bounds, method='L-BFGS-B').fun

    objective_minmax[name] = min_val, max_val
    print(f"  Objective {idx+1}  :    min = {min_val:>15,.1f}  {unit:<12}    max = {max_val:>15,.1f}  {unit}")


  Objective 1  :    min =             0.5  €               max =   368,303,703.7  €
  Objective 2  :    min =             6.5  ships/hour      max =            10.2  ships/hour
  Objective 3  :    min =             6.5  ships/hour      max =            10.2  ships/hour
  Objective 4  :    min =           192.2  complaints/year    max =           745.2  complaints/year
  Objective 5  :    min =           400.0  vehicles/hour    max =         1,600.0  vehicles/hour
